# Chapter 6 — An unbinned likelihood: the structure of Boötes I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/06_likelihood.ipynb)

**Goal.** Once a satellite is found, measure its centre, size, ellipticity and orientation, and the number of its
stars, with a likelihood that uses every star individually — the approach of Martin, de Jong & Rix (2008,
https://arxiv.org/abs/0805.2945), who showed that it works with only tens of member stars. Then check, with
simulations, that the quoted uncertainties are honest, and see why the likelihood-ratio "significance" of a
detection must also be calibrated by simulation.

**You will learn**
1. to write the likelihood of a mixture of a satellite and a uniform background on the sky;
2. the elliptical exponential profile and its normalisation;
3. to estimate uncertainties from the curvature of the log-likelihood, and test their **coverage** with simulations;
4. the test statistic $TS = 2\Delta\ln\mathcal L$ and why $\sqrt{TS}$ is not a Gaussian significance here.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter                        # smoothing of the field CMD histogram
from scipy.optimize import minimize                              # maximum-likelihood fits (Nelder-Mead)
from scipy import stats                                          # chi^2 distribution for the TS comparison

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')        # LVDB v1.1.1 rows: ra, dec [deg], rhalf [arcmin], distance_modulus
iso = load('isochrones_sdss')               # PARSEC isochrones, SDSS absolute magnitudes, one row per initial mass
ref = lvdb.loc['bootes_1']                  # reference centre, r_h, ellipticity, PA and mu of Boötes I

## 6.1 The stars: a CMD mask at the distance of Boötes I

We select stars with the CMD mask of Chapter 5 at $\mu = 19.11$ (the catalogued distance; Chapter 2 measured it),
within 1° of the catalogued centre. Using a mask keeps the data a simple point set on the sky.

In [ ]:
# CMD grid of Chapters 4-5: 40 colour cells of 0.04 mag in (g-r)_0, 65 magnitude cells of 0.1 mag in g_0 (16-22.5)
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')
# Isochrone of 12.6 Gyr (log age 10.1), [Fe/H] = -2.1, sampled at 3000 points equally spaced in CMD path length;
# ISO_W = stars per point from the cumulative IMF (as in Chapter 5)
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
_c, _g = (I.g - I.r).to_numpy(), I.g.to_numpy()
_arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(_c), np.diff(_g)))])
_s = np.linspace(0, _arc[-1], 3000)
ISO_C, ISO_G, ISO_W = np.interp(_s, _arc, _c), np.interp(_s, _arc, _g), np.gradient(np.interp(_s, _arc, I.int_imf))


def satellite_cmd(mu, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Normalised satellite CMD density on the (CC, MM) grid for distance modulus mu (as in Chapters 4-5).
    Colour width: sqrt(2) x the single-band error and 0.03 mag intrinsic; magnitude width: the g error and 0.1 mag
    (one cell); both in quadrature. Returns shape (40, 65), sum 1."""
    pg = ISO_G + mu                                              # apparent g of the isochrone points
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    S = np.einsum('k,kij->ij', ISO_W, np.exp(-0.5 * ((CC - ISO_C[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_mask(S, B, keep=0.5):
    """Cells ranked by S/B; the top ones that maximise sum(S)/sqrt(sum(B)) with sum(S) >= keep (Chapter 5)."""
    order = np.argsort((S / B).ravel())[::-1]                    # cells by decreasing s/b
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])   # signal and background of the top-k cells
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


cat = load('bootes1_sdss')                                       # SDSS DR9 point sources, g < 23, within 2°
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()                # extinction-corrected g and g - r
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
xi, eta = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), ref.ra, ref.dec)   # [deg] about the LVDB centre
# field CMD density from all stars of the 2° file (smoothed by one cell, floored at one star per cell; Chapter 4)
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
M = cmd_mask(satellite_cmd(ref.distance_modulus), B)            # mask at the LVDB mu = 19.11
i, j = np.digitize(c0, cb) - 1, np.digitize(g0, mb) - 1          # CMD cell of every star
# in the mask: on the grid and in a selected cell (np.clip only keeps the index valid for stars off the grid)
inmask = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm)) & M[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)]
RMAX = 1.0                                                        # degrees
use = inmask & (np.hypot(xi, eta) < RMAX)                        # radius 1° = 6 r_h, well inside the 2° file
x, y = xi[use] * 60, eta[use] * 60                                # arcmin
print(f'{use.sum()} stars in the CMD mask within {RMAX}°')

## 6.2 The model

**Satellite.** An elliptical exponential profile with centre $(x_0, y_0)$, half-light radius $r_h$ (along the major
axis), ellipticity $\epsilon = 1 - b/a$ and position angle $\theta$ of the major axis (from north through east). With
$u = (x-x_0)\sin\theta + (y-y_0)\cos\theta$ along the major axis and $v = (x-x_0)\cos\theta - (y-y_0)\sin\theta$
along the minor axis, the elliptical radius is $r_e = \sqrt{u^2 + v^2/(1-\epsilon)^2}$ and

$$ p_{\rm sat}(x, y) = \frac{1}{2\pi a^2 (1-\epsilon)}\, e^{-r_e/a}, \qquad a = r_h / 1.678, $$

normalised over the plane (the part beyond our 60′ circle is $\lesssim 2 \times 10^{-3}$ for $r_h = 10$–$13′$).

**Background.** Uniform over the circle: $p_{\rm bg} = 1/(\pi R_{\max}^2)$.

**Likelihood.** With a satellite fraction $f$, for the $N$ stars,
$\ln\mathcal L = \sum_i \ln[f\, p_{\rm sat}(x_i, y_i) + (1-f)\, p_{\rm bg}]$. The number of satellite stars (in the mask)
is $N_\star = fN$.

One subtlety: if $r_h \to 0$ with the centre on one star, $p_{\rm sat}$ at that star diverges and so does $\mathcal L$.
A mixture likelihood is unbounded; we therefore impose $r_h \ge 0.5′$ (a "satellite" smaller than that is one star).

In [ ]:
R60 = RMAX * 60                                                  # radius of the fitted circle [arcmin]
P_BG = 1 / (np.pi * R60**2)                                      # uniform background density [arcmin^-2]
names = ['x0', 'y0', 'r_h', 'eps', 'theta', 'f']                 # units: arcmin, arcmin, arcmin, -, deg, -


def p_sat(x, y, x0, y0, rh, eps, theta):
    """Elliptical exponential profile [arcmin^-2], normalised over the plane. x, y, x0, y0, rh in arcmin;
    eps = 1 - b/a; theta: PA of the major axis in degrees from north through east."""
    a = rh / 1.678                                               # exponential scale length: r_h = 1.678 a
    t = np.radians(theta)
    u = (x - x0) * np.sin(t) + (y - y0) * np.cos(t)              # along the major axis
    v = (x - x0) * np.cos(t) - (y - y0) * np.sin(t)              # along the minor axis
    return np.exp(-np.sqrt(u**2 + (v / (1 - eps))**2) / a) / (2 * np.pi * a**2 * (1 - eps))


RH_MIN = 0.5                                                     # arcmin: lower bound on r_h (see text)


def unpack(q):
    """Unconstrained vector -> parameters: r_h = RH_MIN + exp, eps and f via logistic functions; theta in [0, 180).
    eps is limited to (0, 0.95) and f to (0, 1), so the optimiser can move freely in q."""
    return q[0], q[1], RH_MIN + np.exp(q[2]), 0.95 / (1 + np.exp(-q[3])), q[4] % 180, 1 / (1 + np.exp(-q[5]))


def nll(q, x, y):
    """-ln L of the mixture f p_sat + (1 - f) p_bg for stars at (x, y) [arcmin]."""
    x0, y0, rh, eps, th, f = unpack(q)
    return -np.sum(np.log(f * p_sat(x, y, x0, y0, rh, eps, th) + (1 - f) * P_BG))


def fit(x, y, start=(0, 0, np.log(8), 0.0, 30.0, -1.0)):
    """Maximum-likelihood fit; several starting position angles guard against local minima.
    start is in the unconstrained q: centre (0, 0), r_h = 0.5 + 8 = 8.5', eps = 0.475, PA = 30°, f = 0.27; fits start
    at PA, PA + 60° and PA + 120° (the PA has period 180°), and the best of the three fits is returned."""
    best = None
    for th0 in (0, 60, 120):
        q0 = np.array(start, float); q0[4] += th0
        r = minimize(nll, q0, args=(x, y), method='Nelder-Mead', options=dict(maxiter=6000, xatol=1e-5, fatol=1e-6))
        if best is None or r.fun < best.fun:
            best = r
    return best


def hessian_errors(q, x, y):
    """1-sigma errors of (x0, y0, r_h, eps, theta, f) from the numerical Hessian of -ln L in physical parameters.
    Returns (p, err): best-fit physical parameters and the square roots of the diagonal of the inverse Hessian
    (marginal errors, correlations included)."""
    p = np.array(unpack(q))
    to_q = lambda p: np.array([p[0], p[1], np.log(p[2] - RH_MIN), -np.log(0.95 / p[3] - 1), p[4], -np.log(1 / p[5] - 1)])
    f = lambda p: nll(to_q(p), x, y)                            # -ln L as a function of the physical parameters
    # finite-difference steps (arcmin, arcmin, 2% of r_h, -, deg, -), several times smaller than the errors found
    # for Boötes I (0.5', 0.7', 1.3', 0.07, 7°, 0.013)
    h = np.array([0.05, 0.05, 0.02 * p[2], 0.01, 1.0, 0.002])
    Hm = np.zeros((6, 6))
    for a in range(6):
        for b in range(6):
            # central-difference second derivative d^2(-ln L) / dp_a dp_b
            ea, eb = np.eye(6)[a] * h[a], np.eye(6)[b] * h[b]
            Hm[a, b] = (f(p + ea + eb) - f(p + ea - eb) - f(p - ea + eb) + f(p - ea - eb)) / (4 * h[a] * h[b])
    return p, np.sqrt(np.diag(np.linalg.inv(Hm)))


best = fit(x, y)
p, err = hessian_errors(best.x, x, y)
lnl0 = len(x) * np.log(P_BG)                                       # f = 0: background only
TS = 2 * (-best.fun - lnl0)                                        # test statistic 2 (ln L_max - ln L_0)
for n, v, e in zip(names, p, err):
    print(f'{n:6s} = {v:8.3f} ± {e:.3f}')
print(f'N_star = {p[5] * len(x):.0f}; TS = {TS:.0f}')              # satellite stars in the mask, within 1°
print(f'LVDB v1.1.1:          r_h = {ref.rhalf:.2f}′, eps = {ref.ellipticity:.2f}, PA = {ref.position_angle:.0f}°')
# Martin, de Jong & Rix (2008, Table 1; https://arxiv.org/abs/0805.2945): the same SDSS data, a similar likelihood
M08 = dict(rh=12.6, rh_err=1.0, eps=0.39, eps_err=0.06, pa=14, pa_err=6)
print(f'Martin et al. (2008): r_h = {M08["rh"]}′ ± {M08["rh_err"]}, eps = {M08["eps"]} ± {M08["eps_err"]}, PA = {M08["pa"]} ± {M08["pa_err"]}°')

Our centre, ellipticity and position angle agree with both references. The half-light radius agrees with Martin et
al. (2008), who fitted the **same SDSS data** with a similar likelihood, but is ~30% (2.5σ) larger than the LVDB
value, taken from the deeper imaging of Muñoz et al. (2018, https://ui.adsabs.harvard.edu/abs/2018ApJ...860...66M; the LVDB
`ref_structure` column). The LVDB documentation does not say whether `rhalf` is a major-axis or circularised radius;
we assume major-axis (UNVERIFIED). A structural parameter is a property
of a *data set and a model*, not only of the object: depth, the CMD selection, the profile family and the treatment of
the outskirts all matter. We therefore test against the same-data measurement, and keep the LVDB comparison as a warning.

In [ ]:
# Figure 6.1, left: the stars and the best-fit ellipses; right: the radial profile and the model
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(x, y, s=2, c='k')
t = np.linspace(0, 2 * np.pi, 200)
x0, y0, rh, eps, th, f = p
for k in (1, 2, 3):
    # ellipse of semi-major axis k r_h and axis ratio 1 - eps, rotated to the PA (inverse of the (u, v) of p_sat)
    u, v = k * rh * np.cos(t), k * rh * (1 - eps) * np.sin(t)
    tr = np.radians(th)
    ax[0].plot(x0 + u * np.sin(tr) + v * np.cos(tr), y0 + u * np.cos(tr) - v * np.sin(tr), c='C3')
ax[0].set(xlim=(30, -30), ylim=(-30, 30), aspect='equal', xlabel="ξ [arcmin]", ylabel="η [arcmin]",
          title='stars in the CMD mask; ellipses at 1, 2, 3 $r_h$')
# radial profile in elliptical annuli, with the model
tr = np.radians(th)
u = (x - x0) * np.sin(tr) + (y - y0) * np.cos(tr)
v = (x - x0) * np.cos(tr) - (y - y0) * np.sin(tr)
re = np.sqrt(u**2 + (v / (1 - eps))**2)                          # elliptical radius of every star [arcmin]
edges = np.arange(0, 45.1, 3)                                    # 3' annuli out to 45' (the 45' ellipse fits in 60')
n = np.histogram(re, edges)[0]
area = np.pi * (1 - eps) * np.diff(edges**2)                     # area of each elliptical annulus [arcmin^2]
rr = 0.5 * (edges[1:] + edges[:-1])
ax[1].errorbar(rr, n / area, np.sqrt(n) / area, fmt='o', c='k', label='data')   # Poisson errors
# model density at the bin centres, evaluated along the major axis (there r_e = rr): satellite + background
ax[1].plot(rr, f * len(x) * p_sat(rr * np.sin(tr) + x0, rr * np.cos(tr) + y0, x0, y0, rh, eps, th) + (1 - f) * len(x) * P_BG, c='C3',
           label='model')
ax[1].axhline((1 - f) * len(x) * P_BG, c='0.5', ls=':', label='background')
ax[1].set(yscale='log', xlabel='elliptical radius [arcmin]', ylabel='stars per arcmin²')
ax[1].legend()
plt.tight_layout()

# Tests (tolerances from STRUCTURE.md):
dpa = abs((p[4] - ref.position_angle + 90) % 180 - 90)          # PA difference folded into [0, 90] (period 180°)
assert np.hypot(p[0], p[1]) < 2                                   # centre within 2' of LVDB
assert abs(p[2] - M08['rh']) < 2 * np.hypot(err[2], M08['rh_err'])  # r_h within 2 combined sigma of Martin+08
# ellipticity within 0.15 of both references (about twice our error, 0.07); PA within 15° of LVDB (about twice 7°)
assert abs(p[3] - ref.ellipticity) < 0.15 and abs(p[3] - M08['eps']) < 0.15
assert dpa < 15

**Figure 6.1.** Left: positions of the 2596 SDSS point sources (extinction-corrected) in the CMD mask at μ = 19.11
within 1° of Boötes I (black dots), in tangent-plane coordinates ξ (east to the left) and η in arcmin about the LVDB
centre; only the central ±30′ is shown. Red: the best-fitting elliptical exponential at 1, 2 and 3 $r_h$ (semi-major
axes 13.2′, 26.4′, 39.6′; ε = 0.38, θ = 5°). Right: star density (stars per arcmin², log scale) in 3′-wide elliptical
annuli of the best-fit shape, out to an elliptical radius of 45′ (black points, Poisson errors); red line: the model
(satellite of $N_\star$ = 397 stars + uniform background) at the bin centres; grey dotted line: the fitted background,
0.19 stars per arcmin². The exponential profile describes Boötes I until it sinks into the background at ~30′
(≈ 2.3 $r_h$).

## 6.3 Are the error bars honest? A coverage test

The errors above come from the curvature of $\ln\mathcal L$ at its maximum, which is exact only for a Gaussian
likelihood. We test them: simulate 200 data sets from the best-fit model (same total N, satellite stars drawn from
the profile, background uniform), fit each, and count how often the true value lies within the quoted 1σ error.
For honest errors this happens in 68% of the simulations (with 200 simulations, the binomial scatter is ±3.3%).

In [ ]:
rng = np.random.default_rng(6)                                    # fixed seed: the simulations are reproducible


def simulate(p, n):
    """Positions of n stars drawn from the model with parameters p (arcmin).
    The number of satellite stars is binomial(n, f); satellite stars falling outside the 60' circle are dropped
    and the rest of the n stars are drawn uniformly in the circle, so the total is always n."""
    x0, y0, rh, eps, th, f = p
    ns = rng.binomial(n, f)
    a = rh / 1.678
    r = rng.gamma(2, a, ns)                                       # exponential surface density: r ~ Gamma(2, a)
    phi = rng.uniform(0, 2 * np.pi, ns)
    u, v = r * np.cos(phi), r * np.sin(phi) * (1 - eps)          # squash the minor axis by 1 - eps
    tr = np.radians(th)
    xs, ys = x0 + u * np.sin(tr) + v * np.cos(tr), y0 + u * np.cos(tr) - v * np.sin(tr)
    keep = np.hypot(xs, ys) < R60
    # uniform background in the circle: radius R sqrt(U) gives a uniform density per unit area
    rb, pb = R60 * np.sqrt(rng.uniform(0, 1, n - keep.sum())), rng.uniform(0, 2 * np.pi, n - keep.sum())
    return np.concatenate([xs[keep], rb * np.cos(pb)]), np.concatenate([ys[keep], rb * np.sin(pb)])


# 200 simulations of the best-fit model with the observed number of stars (2596); each fit starts at the centre
# (0, 0) and the true r_h, PA and f, with eps = 0.475 (fit adds starts at PA + 60° and PA + 120°)
cover = []
for k in range(200):
    xs, ys = simulate(p, len(x))
    b = fit(xs, ys, start=(0, 0, np.log(p[2] - RH_MIN), 0.0, p[4], best.x[5]))
    ps, es = hessian_errors(b.x, xs, ys)
    d = ps - p                                                    # fitted - true, per parameter
    d[4] = (d[4] + 90) % 180 - 90                                 # position angle is defined modulo 180
    cover.append(np.abs(d) < es)                                  # is the truth inside the quoted 1-sigma interval?
cover = np.array(cover).mean(0)                                   # fraction of simulations, per parameter
for n_, c in zip(names, cover):
    print(f'{n_:6s}: true value within 1 sigma in {c:.2f} of simulations')

# Figure 6.2: coverage per parameter, with the band expected for honest errors
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(names, cover)
# binomial scatter of a fraction 0.68 from 200 trials: sqrt(0.68 x 0.32 / 200) = 0.033
ax.axhspan(0.68 - 2 * 0.033, 0.68 + 2 * 0.033, color='C1', alpha=0.3, label='0.68 ± 2σ (binomial, 200 sims)')
ax.set(ylabel='coverage of the 1σ interval', ylim=(0, 1))
ax.legend()
plt.tight_layout()
# test: r_h coverage in 60-76% (STRUCTURE.md), i.e. 0.68 within about +-2.4 binomial sigma
assert 0.60 < cover[2] < 0.76                                     # the half-light radius

**Figure 6.2.** Coverage of the 1σ Hessian errors: for each parameter (blue bars), the fraction of 200 simulated data
sets — 2596 stars drawn from the best-fit model of Figure 6.1, refitted — in which the true value lies within the
fitted value ± its Hessian error. Orange band: 0.68 ± 2 binomial σ (0.68 ± 0.066) expected for honest errors.
Coverage: $x_0$ 0.60, $y_0$ 0.66, $r_h$ 0.66, ε 0.64, θ 0.66, $f$ 0.59. With ~400 satellite stars the likelihood is
close to Gaussian: the errors of $r_h$, ε, θ and $y_0$ are honest (0.64–0.66), while those of the satellite fraction
and of $x_0$ are slightly optimistic (0.59–0.60, about 2.5 binomial σ below 0.68). For a satellite with tens of stars
the Hessian would be far worse: then use the profile likelihood or a posterior sample (e.g. MCMC).

## 6.4 How significant is a detection? Why not $\sqrt{TS}$

The test statistic compares the best satellite model with the background-only model, $TS = 2[\ln\mathcal L_{\max} -
\ln\mathcal L_0]$. For Boötes I, TS ≈ 390: no doubt. For a marginal candidate one is tempted to
quote $\sqrt{TS}$ as "the number of sigmas" (Wilks' theorem: TS ~ $\chi^2_k$ under the null). Wilks' theorem does
not apply here: $f \ge 0$ sits on the boundary of the parameter space at the null, and the centre, size, ellipticity
and angle are *undefined* when $f = 0$. Fitting pure-background simulations shows what TS looks like with no
satellite.

In [ ]:
# 100 background-only data sets (f = 0, so the other entries of p are irrelevant), each with the observed number
# of stars, fitted with the full satellite model. Each fit starts from 3 random centres in the 80' x 80' box around
# the centre (x 3 starting PAs inside fit), with r_h = 4.5' and f = 0.05, and the best is kept.
ts_null = []
for k in range(100):
    xs, ys = simulate((0, 0, 8, 0.3, 0, 0.0), len(x))             # f = 0: background only
    b = min((fit(xs, ys, start=(sx, sy, np.log(4), 0, 0, -3)) for sx, sy in rng.uniform(-40, 40, (3, 2))), key=lambda r: r.fun)
    ts_null.append(2 * (-b.fun - len(xs) * np.log(P_BG)))        # TS against the background-only likelihood
ts_null = np.array(ts_null)
print(f'null TS: median {np.median(ts_null):.1f}, 95th percentile {np.percentile(ts_null, 95):.1f}; '
      f'chi2 with 1 dof would give 95th percentile {stats.chi2.ppf(0.95, 1):.1f}')
# Figure 6.3: histogram of the null TS (20 bins, normalised to unit area) against the chi^2_1 density
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(ts_null, 20, density=True, label='fits to pure-background simulations')
tt = np.linspace(0.01, ts_null.max(), 200)                        # start above 0, where the chi^2_1 density diverges
ax.plot(tt, stats.chi2.pdf(tt, 1), label=r'$\chi^2_1$')
ax.set(xlabel='TS', ylabel='density', yscale='log')               # log: the chi^2_1 density diverges at TS -> 0
ax.legend()
plt.tight_layout()

**Figure 6.3.** Blue: histogram (20 bins, normalised to unit area) of TS for 100 background-only simulations (2596
stars uniform in the 60′ circle), each fitted with the full six-parameter satellite model from only a few starting
points (3 random centres × 3 position angles; a real search would scan the whole field, making the null TS larger
still). Orange: the $\chi^2_1$ density, which diverges at TS = 0. The vertical axis is logarithmic so that both are
visible. The null TS lie between ≈ 9 and 32 (median 19.0, 95th percentile 25.8), against a 95th percentile of 3.8 for
$\chi^2_1$: a TS of 10 is not "3σ". As in Chapter 5, significance must be calibrated by simulating the whole
procedure.

## Summary

- A mixture of an elliptical exponential and a uniform background, fitted star by star, measures the structure of
  Boötes I; it reproduces the SDSS measurement of Martin et al. (2008). Deeper data give a smaller $r_h$: compare
  measurements made with matching data and definitions.
- Hessian errors are close to honest here (coverage 0.59–0.66) because the satellite has hundreds of stars; check
  this, don't assume it.
- $TS$ does not follow $\chi^2$ for detections; calibrate with simulations.